# Treinamento modelo deduplicação base cnefe

## 0 - Setup

### Configura Paths

In [1]:
from pathlib import Path
import getpass
import os

USER_NAME = getpass.getuser()
PATH_HOME = Path.home()
PROJECT_ROOT_PATH = PATH_HOME / "projects/brazilian_address_linkage"
DATALAKE_PATH = Path(f"/media/{USER_NAME}/Seagate Portable Drive/Datalake/")
SALTING_PARTITIONS = 100

os.chdir(PROJECT_ROOT_PATH)

### Carrega Dados Limpos

In [2]:
from pyspark import SparkConf, SparkContext
from pyspark.sql import SparkSession
from splink.backends.spark import similarity_jar_location
from splink import SparkAPI
from splink import block_on
from pathlib import Path
import warnings
import os

warnings.filterwarnings("ignore")

COD_MUNINCIPIO_CAPITAL_RJ = "3304557"

conf = SparkConf()
conf.set("spark.driver.memory", "20g")
conf.set("spark.worker.memory", "6g")
conf.set("spark.master", "local[4]")
conf.set("spark.local.dir", "spark_tmp/")
conf.set("spark.sql.shuffle.partitions", "600")
conf.set("spark.shuffle.checksum.enabled", "true")
conf.set("spark.shuffle.detectCorrupt", "true")
path = similarity_jar_location()
conf.set("spark.jars", path)

sc = SparkContext.getOrCreate(conf=conf)

spark = SparkSession(sc)
spark.sparkContext.setCheckpointDir("./tmp_checkpoints")
db_api = SparkAPI(spark_session=spark)
cnefe_df = spark.read.parquet(
    os.path.join(DATALAKE_PATH, "silver/cnefe/cleaned_cnefe_addresses.parquet")
)
cnefe_rj_df = cnefe_df.filter(cnefe_df.COD_MUNICIPIO == COD_MUNINCIPIO_CAPITAL_RJ)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/10/08 20:13:07 WARN Utils: Your hostname, lucas-vital-Q570M-D3H, resolves to a loopback address: 127.0.1.1; using 192.168.100.25 instead (on interface wlp8s0)
26/10/08 20:13:07 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
26/10/08 20:13:08 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/08 20:13:08 WARN SparkConf: Note that spark.local.dir will be overridden by the value set by the cluster manager (via SPARK_LOCAL_DIRS in standalone/kubernetes and LOCAL_DIRS in YARN).


In [3]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

w = Window.orderBy(F.monotonically_increasing_id())

cnefe_rj_df = cnefe_rj_df.withColumn(
    "unique_id",
    F.concat(F.lit("CNEFE"), F.lpad(F.row_number().over(w).cast("string"), 10, "0")),
)


In [4]:
cnefe_rj_df.show(5)

26/10/08 20:13:12 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/08 20:13:12 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/08 20:13:12 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/08 20:13:12 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+------------------+------+-------------+------------+---------------+----------------+----------+--------+--------+--------------+----------------+------------------+-----------+------------+---------------+--------------+--------------+--------------+--------------+--------------+--------------+--------------+--------------+--------------+--------------+----------+----------+------------+-----------+--------------------+----------------------------+----------------------------+------------------------------+---------------+---+--------------------+----------------+-------------------+----------------------+---------------+
|COD_UNICO_ENDERECO|COD_UF|COD_MUNICIPIO|COD_DISTRITO|COD_SUBDISTRITO|       COD_SETOR|NUM_QUADRA|NUM_FACE|     CEP|DSC_LOCALIDADE|NOM_TIPO_SEGLOGR|NOM_TITULO_SEGLOGR|NOM_SEGLOGR|NUM_ENDERECO|DSC_MODIFICADOR|NOM_COMP_ELEM1|VAL_COMP_ELEM1|NOM_COMP_ELEM2|VAL_COMP_ELEM2|NOM_COMP_ELEM3|VAL_COMP_ELEM3|NOM_COMP_ELEM4|VAL_COMP_ELEM4|NOM_COMP_ELEM5|VAL_COMP_ELEM5|  LATITUD

## Definição parâmetros de treinamento

### Escolhendo níveis de comparação

In [5]:
LINKAGE_COLS = [
    "COD_UNICO_ENDERECO",
    "COD_MUNICIPIO",
    "COD_DISTRITO",
    "COD_SUBDISTRITO",
    "COD_SETOR",
    "NUM_QUADRA",
    "NUM_FACE",
    "CEP",
    "DSC_LOCALIDADE",
    "NOM_TIPO_SEGLOGR",
    "NOM_SEGLOGR",
    "NUM_ENDERECO",
    "LATITUDE",
    "LONGITUDE",
    "NV_GEO_COORD",
    "COD_ESPECIE",
    "UF",
    "COD_TIPO_ESPECI",
    "ENDERECO_COMPLETO",
    "NOM_SEGLOGR_phon",
    "DSC_LOCALIDADE_phon",
    "ENDERECO_COMPLETO_phon",
]

In [6]:
def jaccard_similarity(str1, str2):
    set1 = set(str1)
    set2 = set(str2)
    return len(set1 & set2) / len(set1 | set2)

In [7]:
import splink.comparison_library as cl

JAIRO_WINKLER_THRESHOLDS = [0.95, 0.90, 0.8]
KM_DISTANCES_THRESHOLDS = [4, 2, 1]
LEVEINTHEIN_THRESHOLDS = [0, 1]


comparison_levels = [
    cl.ExactMatch(col_name="COD_UNICO_ENDERECO"),
    cl.ExactMatch(col_name="COD_MUNICIPIO").configure(term_frequency_adjustments=True),
    cl.ExactMatch(col_name="COD_DISTRITO").configure(term_frequency_adjustments=True),
    cl.ExactMatch(col_name="COD_SUBDISTRITO"),
    cl.ExactMatch(col_name="COD_SETOR"),
    cl.DistanceInKMAtThresholds(
        lat_col="LATITUDE", long_col="LONGITUDE", km_thresholds=KM_DISTANCES_THRESHOLDS
    ),
    cl.LevenshteinAtThresholds(
        col_name="NUM_QUADRA", distance_threshold_or_thresholds=LEVEINTHEIN_THRESHOLDS
    ).configure(term_frequency_adjustments=True),
    cl.NameComparison(
        col_name="NOM_TIPO_SEGLOGR", jaro_winkler_thresholds=JAIRO_WINKLER_THRESHOLDS
    ).configure(term_frequency_adjustments=True),
    cl.LevenshteinAtThresholds(
        col_name="NUM_ENDERECO", distance_threshold_or_thresholds=LEVEINTHEIN_THRESHOLDS
    ).configure(term_frequency_adjustments=True),
    cl.ExactMatch(col_name="NUM_FACE").configure(term_frequency_adjustments=True),
    cl.LevenshteinAtThresholds(
        col_name="CEP", distance_threshold_or_thresholds=[0, 1, 2]
    ),
    cl.NameComparison(
        col_name="DSC_LOCALIDADE", jaro_winkler_thresholds=JAIRO_WINKLER_THRESHOLDS
    ),
    cl.NameComparison(
        col_name="NOM_SEGLOGR", jaro_winkler_thresholds=JAIRO_WINKLER_THRESHOLDS
    ),
    cl.ExactMatch(col_name="NV_GEO_COORD").configure(term_frequency_adjustments=True),
    cl.ExactMatch(col_name="COD_ESPECIE").configure(term_frequency_adjustments=True),
    # cl.LevenshteinAtThresholds(
    #     col_name="COD_ESPECIE", distance_threshold_or_thresholds=LEVEINTHEIN_THRESHOLDS
    # ).configure(term_frequency_adjustments=True),
    cl.NameComparison(
        col_name="ENDERECO_COMPLETO", jaro_winkler_thresholds=JAIRO_WINKLER_THRESHOLDS
    ),
]


## Treinamento

### Estimativa de random match probability

In [8]:
from splink import Linker, SettingsCreator

RULES_FOR_INFERENCE=[
    block_on("COD_UNICO_ENDERECO"),
    block_on("ENDERECO_COMPLETO_phon","DSC_LOCALIDADE_phon","COD_MUNICIPIO", "DSC_LOCALIDADE_phon","CEP", salting_partitions=SALTING_PARTITIONS),
    block_on("ENDERECO_COMPLETO_phon","DSC_LOCALIDADE_phon","COD_MUNICIPIO", "DSC_LOCALIDADE_phon","CEP", "COD_SETOR", salting_partitions=SALTING_PARTITIONS),
    block_on("ENDERECO_COMPLETO_phon","DSC_LOCALIDADE_phon","COD_MUNICIPIO", "DSC_LOCALIDADE_phon","CEP", "COD_SETOR", salting_partitions=SALTING_PARTITIONS),
    block_on("ENDERECO_COMPLETO_phon","DSC_LOCALIDADE_phon","COD_MUNICIPIO", "DSC_LOCALIDADE_phon","NOM_SEGLOGR_phon","CEP", salting_partitions=SALTING_PARTITIONS),
    block_on("ENDERECO_COMPLETO_phon","DSC_LOCALIDADE_phon","COD_MUNICIPIO", "DSC_LOCALIDADE_phon","CEP", "COD_SETOR", salting_partitions=SALTING_PARTITIONS),
    block_on("ENDERECO_COMPLETO_phon","DSC_LOCALIDADE_phon","COD_MUNICIPIO", "DSC_LOCALIDADE_phon","NOM_SEGLOGR_phon", salting_partitions=SALTING_PARTITIONS),
    block_on("ENDERECO_COMPLETO_phon","DSC_LOCALIDADE_phon","COD_MUNICIPIO", "DSC_LOCALIDADE_phon", salting_partitions=SALTING_PARTITIONS),
    block_on("ENDERECO_COMPLETO_phon", "COD_MUNICIPIO","COD_SETOR", salting_partitions=SALTING_PARTITIONS),
    block_on("ENDERECO_COMPLETO_phon","COD_MUNICIPIO", "DSC_LOCALIDADE_phon", salting_partitions=SALTING_PARTITIONS),
    block_on("ENDERECO_COMPLETO_phon","DSC_LOCALIDADE_phon","COD_MUNICIPIO", salting_partitions=SALTING_PARTITIONS), 
    block_on("ENDERECO_COMPLETO_phon","COD_MUNICIPIO", "DSC_LOCALIDADE_phon", "COD_ESPECIE", salting_partitions=SALTING_PARTITIONS),
]
linker_settings = SettingsCreator(
    link_type="dedupe_only",
    comparisons=comparison_levels,
    blocking_rules_to_generate_predictions=RULES_FOR_INFERENCE
   
)
linker = Linker(cnefe_rj_df, settings=linker_settings, db_api=db_api)


### Estimating  two random probability ($\lambda$)

In [9]:
deterministic_rules = ["l.ENDERECO_COMPLETO_phon = r.ENDERECO_COMPLETO_phon"]

linker.training.estimate_probability_two_random_records_match(
    deterministic_rules, recall=0.7
)


26/10/08 20:13:28 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/08 20:13:28 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/08 20:13:28 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/08 20:13:28 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/08 20:13:28 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/08 20:13:28 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/08 2

### Estimate the probabilithy of match among non matches ($\mu$)

In [10]:
linker.training.estimate_u_using_random_sampling(max_pairs=1e8)

----- Estimating u probabilities using random sampling -----
26/10/08 20:13:37 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/08 20:13:37 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/08 20:13:37 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/08 20:13:42 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/08 20:13:42 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/08 20:13:47 WARN DataSource: All paths were ignored:                      
  file:/home/lucas-vital/projects

### Estimating probability of match among true matches (m)

In [11]:

linker.training.estimate_parameters_using_expectation_maximisation(
    blocking_rule=block_on(
        "ENDERECO_COMPLETO_phon",
        "COD_MUNICIPIO",
        "DSC_LOCALIDADE_phon",
        "COD_ESPECIE",
    ),
    
)

26/10/08 20:25:15 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/08 20:25:15 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/08 20:25:15 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/08 20:25:15 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/08 20:25:15 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/08 20:25:15 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/08 2

<EMTrainingSession, blocking on (l.`ENDERECO_COMPLETO_phon` = r.`ENDERECO_COMPLETO_phon`) AND (l.`COD_MUNICIPIO` = r.`COD_MUNICIPIO`) AND (l.`DSC_LOCALIDADE_phon` = r.`DSC_LOCALIDADE_phon`) AND (l.`COD_ESPECIE` = r.`COD_ESPECIE`), deactivating comparisons COD_MUNICIPIO, COD_ESPECIE>

In [12]:

linker.training.estimate_parameters_using_expectation_maximisation(
    blocking_rule=block_on(
        "ENDERECO_COMPLETO_phon",
        "COD_MUNICIPIO",
        "COD_SETOR",
    )
)


----- Starting EM training session -----

Estimating the m probabilities of the model by blocking on:
(l.`ENDERECO_COMPLETO_phon` = r.`ENDERECO_COMPLETO_phon`) AND (l.`COD_MUNICIPIO` = r.`COD_MUNICIPIO`) AND (l.`COD_SETOR` = r.`COD_SETOR`)

Parameter estimates will be made for the following comparison(s):
    - COD_UNICO_ENDERECO
    - COD_DISTRITO
    - COD_SUBDISTRITO
    - LATITUDE_LONGITUDE
    - NUM_QUADRA
    - NOM_TIPO_SEGLOGR
    - NUM_ENDERECO
    - NUM_FACE
    - CEP
    - DSC_LOCALIDADE
    - NOM_SEGLOGR
    - NV_GEO_COORD
    - COD_ESPECIE
    - ENDERECO_COMPLETO

Parameter estimates cannot be made for the following comparison(s) since they are used in the blocking rules: 
    - COD_MUNICIPIO
    - COD_SETOR
26/10/08 20:28:29 WARN DataSource: All paths were ignored:                      
  file:/home/lucas-vital/projects/brazilian_address_linkage/tmp_checkpoints/26c4fc6d-a833-49b7-a192-3b11beb4d119/__splink__blocked_id_pairs_2f58c5b56
26/10/08 20:28:47 WARN DataSource: All

<EMTrainingSession, blocking on (l.`ENDERECO_COMPLETO_phon` = r.`ENDERECO_COMPLETO_phon`) AND (l.`COD_MUNICIPIO` = r.`COD_MUNICIPIO`) AND (l.`COD_SETOR` = r.`COD_SETOR`), deactivating comparisons COD_MUNICIPIO, COD_SETOR>

In [13]:


linker.training.estimate_parameters_using_expectation_maximisation(
    blocking_rule=block_on(
        "ENDERECO_COMPLETO_phon",
        "DSC_LOCALIDADE_phon",  
        "COD_MUNICIPIO",
        "CEP",
        "COD_SETOR",
    )
)


----- Starting EM training session -----

Estimating the m probabilities of the model by blocking on:
(l.`ENDERECO_COMPLETO_phon` = r.`ENDERECO_COMPLETO_phon`) AND (l.`DSC_LOCALIDADE_phon` = r.`DSC_LOCALIDADE_phon`) AND (l.`COD_MUNICIPIO` = r.`COD_MUNICIPIO`) AND (l.`CEP` = r.`CEP`) AND (l.`COD_SETOR` = r.`COD_SETOR`)

Parameter estimates will be made for the following comparison(s):
    - COD_UNICO_ENDERECO
    - COD_DISTRITO
    - COD_SUBDISTRITO
    - LATITUDE_LONGITUDE
    - NUM_QUADRA
    - NOM_TIPO_SEGLOGR
    - NUM_ENDERECO
    - NUM_FACE
    - DSC_LOCALIDADE
    - NOM_SEGLOGR
    - NV_GEO_COORD
    - COD_ESPECIE
    - ENDERECO_COMPLETO

Parameter estimates cannot be made for the following comparison(s) since they are used in the blocking rules: 
    - COD_MUNICIPIO
    - COD_SETOR
    - CEP
26/10/08 20:30:30 WARN DataSource: All paths were ignored:                      
  file:/home/lucas-vital/projects/brazilian_address_linkage/tmp_checkpoints/26c4fc6d-a833-49b7-a192-3b11beb4

<EMTrainingSession, blocking on (l.`ENDERECO_COMPLETO_phon` = r.`ENDERECO_COMPLETO_phon`) AND (l.`DSC_LOCALIDADE_phon` = r.`DSC_LOCALIDADE_phon`) AND (l.`COD_MUNICIPIO` = r.`COD_MUNICIPIO`) AND (l.`CEP` = r.`CEP`) AND (l.`COD_SETOR` = r.`COD_SETOR`), deactivating comparisons COD_MUNICIPIO, COD_SETOR, CEP>

In [14]:
linker.training.estimate_parameters_using_expectation_maximisation(
    blocking_rule=block_on(
        "ENDERECO_COMPLETO_phon",
        "COD_MUNICIPIO",
        "COD_SETOR",
    )
)


----- Starting EM training session -----

Estimating the m probabilities of the model by blocking on:
(l.`ENDERECO_COMPLETO_phon` = r.`ENDERECO_COMPLETO_phon`) AND (l.`COD_MUNICIPIO` = r.`COD_MUNICIPIO`) AND (l.`COD_SETOR` = r.`COD_SETOR`)

Parameter estimates will be made for the following comparison(s):
    - COD_UNICO_ENDERECO
    - COD_DISTRITO
    - COD_SUBDISTRITO
    - LATITUDE_LONGITUDE
    - NUM_QUADRA
    - NOM_TIPO_SEGLOGR
    - NUM_ENDERECO
    - NUM_FACE
    - CEP
    - DSC_LOCALIDADE
    - NOM_SEGLOGR
    - NV_GEO_COORD
    - COD_ESPECIE
    - ENDERECO_COMPLETO

Parameter estimates cannot be made for the following comparison(s) since they are used in the blocking rules: 
    - COD_MUNICIPIO
    - COD_SETOR

26/10/08 20:32:12 WARN DAGScheduler: Broadcasting large task binary with size 1728.6 KiB
Level All other comparisons on comparison COD_DISTRITO not observed in dataset, unable to train m value

Level All other comparisons on comparison COD_SUBDISTRITO not observed in 

<EMTrainingSession, blocking on (l.`ENDERECO_COMPLETO_phon` = r.`ENDERECO_COMPLETO_phon`) AND (l.`COD_MUNICIPIO` = r.`COD_MUNICIPIO`) AND (l.`COD_SETOR` = r.`COD_SETOR`), deactivating comparisons COD_MUNICIPIO, COD_SETOR>

In [15]:
linker.training.estimate_parameters_using_expectation_maximisation(
    blocking_rule=block_on(
        "COD_UNICO_ENDERECO",
    )
)


----- Starting EM training session -----



Estimating the m probabilities of the model by blocking on:
l.`COD_UNICO_ENDERECO` = r.`COD_UNICO_ENDERECO`

Parameter estimates will be made for the following comparison(s):
    - COD_MUNICIPIO
    - COD_DISTRITO
    - COD_SUBDISTRITO
    - COD_SETOR
    - LATITUDE_LONGITUDE
    - NUM_QUADRA
    - NOM_TIPO_SEGLOGR
    - NUM_ENDERECO
    - NUM_FACE
    - CEP
    - DSC_LOCALIDADE
    - NOM_SEGLOGR
    - NV_GEO_COORD
    - COD_ESPECIE
    - ENDERECO_COMPLETO

Parameter estimates cannot be made for the following comparison(s) since they are used in the blocking rules: 
    - COD_UNICO_ENDERECO
26/10/08 20:33:18 WARN DataSource: All paths were ignored:                      
  file:/home/lucas-vital/projects/brazilian_address_linkage/tmp_checkpoints/26c4fc6d-a833-49b7-a192-3b11beb4d119/__splink__blocked_id_pairs_e813bb71f
26/10/08 20:33:24 WARN DataSource: All paths were ignored:                      
  file:/home/lucas-vital/projects/brazilian_address_linkage/tmp_checkpoints/26c4fc6d-a833-

<EMTrainingSession, blocking on l.`COD_UNICO_ENDERECO` = r.`COD_UNICO_ENDERECO`, deactivating comparisons COD_UNICO_ENDERECO>

## Visualising model parameters

In [16]:
linker.visualisations.match_weights_chart()

alt.VConcatChart(...)

In [17]:
linker.visualisations.m_u_parameters_chart()

alt.HConcatChart(...)

In [18]:
linker.visualisations.parameter_estimate_comparisons_chart()

alt.Chart(...)

## Exporting the generated model

In [19]:
Path("splink_models/").mkdir(parents=True, exist_ok=True)
settings = linker.misc.save_model_to_json(
    "splink_models/cnefe_dedup_model.json", overwrite=True
)